# OVD-Diagnose — Domain 3: Underwater (Brackish)

**Purpose: test the intermediate-difficulty hypothesis.** Aerial is vocabulary-bound
(AP 0.015, AR_SAM 0.138); medical is degenerate (AP 0.0002, AR_SAM 0.0024 — everything
fails at once, so the axes cannot separate). A benchmark claim needs a domain *between*
them, where AP is low but non-vanishing and the three axes stay individually readable.

Brackish should be that point: classes are common nouns (`fish, crab, shrimp, jellyfish,
starfish`) that CLIP-family text encoders know well, but the imagery is turbid low-visibility
water. Known vocabulary + hard optics = S measurable, L non-trivial.

**Falsifiable prediction (recorded before running):** OWLv2 `AP_global` in **0.03–0.12**,
`AR_SAM` well above medical's 0.002. If AP lands ≈0, the hypothesis is wrong, underwater is a
second degenerate domain, and we drop it rather than pad the table.

**Data:** Add Data → `aalborguniversity/brackish-dataset`. GPU + Internet on.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true
!pip install -q ultralytics transformers pycocotools pyyaml pillow

## Layout discovery
Kaggle mirrors repackage this dataset differently. Look before converting — the VinDr
box-scale bug came from assuming a layout.

In [ ]:
import glob, os
print('--- inputs ---')
for p in sorted(glob.glob('/kaggle/input/*')): print(' ', p)
print('\n--- dirs (depth<=4) ---')
!find /kaggle/input -maxdepth 4 -type d | head -40
print('\n--- sample label files ---')
labs = [p for p in glob.glob('/kaggle/input/**/*.txt', recursive=True)
        if 'label' in p.lower() or os.path.basename(os.path.dirname(p)).lower() == 'labels'][:3]
if not labs:
    labs = glob.glob('/kaggle/input/**/*.txt', recursive=True)[:3]
for l in labs:
    print(l)
    print('   ', open(l).read()[:180].replace('\n', ' | '))
print('\n--- yaml/names ---')
for p in glob.glob('/kaggle/input/**/*.yaml', recursive=True)[:3] + \
         glob.glob('/kaggle/input/**/classes.txt', recursive=True)[:3]:
    print(p); print('   ', open(p).read()[:300].replace('\n', ' | '))

In [ ]:
# Auto-pick root = dir containing images/ + labels/. Override manually if wrong.
ROOT, SPLIT = '', ''          # <- set by hand if autodetect misses
if not ROOT:
    for d in sorted(glob.glob('/kaggle/input/**/', recursive=True)):
        if os.path.isdir(os.path.join(d, 'images')) and os.path.isdir(os.path.join(d, 'labels')):
            ROOT = d.rstrip('/'); break
    # split-style layout: root/valid/{images,labels}
    if not ROOT:
        for d in sorted(glob.glob('/kaggle/input/**/', recursive=True)):
            for s in ('valid', 'val', 'test', 'train'):
                if os.path.isdir(os.path.join(d, s, 'images')):
                    ROOT, SPLIT = d.rstrip('/'), s; break
            if ROOT: break
assert ROOT, 'set ROOT manually from the listing above'
IMG_DIR = os.path.join(ROOT, SPLIT, 'images') if SPLIT else os.path.join(ROOT, 'images')
if not os.path.isdir(IMG_DIR):
    IMG_DIR = os.path.join(ROOT, SPLIT) if SPLIT else ROOT
print('ROOT   =', ROOT)
print('SPLIT  =', SPLIT or '(none)')
print('IMG_DIR=', IMG_DIR, '| exists:', os.path.isdir(IMG_DIR))

In [ ]:
OUT = 'data/underwater/annotations/instances_val.json'
split_arg = f'--split {SPLIT}' if SPLIT else ''
!python tools/yolo_to_coco.py --root "{ROOT}" {split_arg} --out "{OUT}"
# converter prints a WARN if boxes were already absolute — do not proceed past a WARN
from pycocotools.coco import COCO
c = COCO(OUT)
print('classes:', [x['name'] for x in c.loadCats(sorted(c.getCatIds()))])
print('images:', len(c.imgs), '| anns:', len(c.anns))

## Run all models
`LIMIT = 300` first — enough to test the intermediate-regime prediction cheaply.
Set `LIMIT = 0` for the full split only after the verdict cell says the domain is usable.

In [ ]:
LIMIT = 300
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
!python tools/run_all.py \
  --ann  data/underwater/annotations/instances_val.json \
  --imgs "{IMG_DIR}" \
  --out  runs/diag/underwater \
  --device cuda:0 {limit_arg} --bootstrap 1000 \
  --models "yoloworld:yolov8s-world.pt,owlv2:google/owlv2-base-patch16-ensemble,groundingdino:IDEA-Research/grounding-dino-tiny" \
  --sam_weights mobile_sam.pt

In [ ]:
import pandas as pd
print(pd.read_csv('runs/diag/underwater/fingerprints.csv').to_string(index=False))

## Verdict — is this domain non-degenerate?
The whole reason for a third domain. Decide here, before investing in controls.

In [ ]:
import pandas as pd
fp = pd.read_csv('runs/diag/underwater/fingerprints.csv').set_index('model')
print(f"{'domain':<12}{'AP_global':>11}{'AP_oracle':>11}{'AR_SAM':>9}{'L_det':>8}")
print(f"{'aerial(ref)':<12}{0.0150:>11.4f}{0.0787:>11.4f}{0.1376:>9.4f}{0.835:>8.3f}")
print(f"{'medical(ref)':<12}{0.0002:>11.4f}{0.0012:>11.4f}{0.0024:>9.4f}{0.943:>8.3f}")
for m in fp.index:
    r = fp.loc[m]
    print(f"{m:<12}{r.AP_global:>11.4f}{r.AP_oracle:>11.4f}{r.AR_agnostic:>9.4f}{r.L_det:>8.3f}")

ap  = float(fp.loc['owlv2', 'AP_global'])
apo = float(fp.loc['owlv2', 'AP_oracle'])
print('\n--- verdict (OWLv2) ---')
if ap >= 0.03 and apo >= 0.05:
    print('NON-DEGENERATE. AP is comfortably above the medical floor and oracle AP leaves')
    print('headroom, so S_norm is a real ratio and all three axes are individually readable.')
    print('-> Keep domain. Rerun with LIMIT=0, then run the controls below.')
elif ap > 0.005:
    print('WEAKLY NON-DEGENERATE. Usable, but S rests on a small AP_oracle denominator;')
    print('report S with the same instability caveat used for medical.')
    print('-> Keep domain, but do not lead with the S axis here.')
else:
    print('DEGENERATE — a second medical-like domain, prediction falsified.')
    print('-> Do NOT pad the table with a third all-zero row. The honest conclusion becomes')
    print('   that OVD failure in specialized domains is more uniformly catastrophic than the')
    print('   aerial/medical contrast suggested. Report that instead.')

## Calibration + qualitative

In [ ]:
!python tools/plot_reliability.py \
  --ann data/underwater/annotations/instances_val.json \
  --models yoloworld:runs/diag/underwater/yoloworld/results_global.json \
           owlv2:runs/diag/underwater/owlv2/results_global.json \
           groundingdino:runs/diag/underwater/groundingdino/results_global.json \
  --out paper/figures/reliability_underwater

In [ ]:
# Both modes: we do not know a priori which failure dominates here (that is the point).
for mode in ['localized_wrong', 'missed']:
    !python tools/qualitative.py \
      --ann data/underwater/annotations/instances_val.json --imgs "{IMG_DIR}" \
      --results runs/diag/underwater/owlv2/results_global.json \
      --mode {mode} --n 6 --out paper/figures/qual_underwater_{mode}.png
from IPython.display import Image, display
for mode in ['localized_wrong', 'missed']:
    print(mode); display(Image(f'paper/figures/qual_underwater_{mode}.png'))

## Validation controls
Run only if the verdict cell said non-degenerate. On a domain with real AP these carry
more weight than the aerial runs, because none of the three axes sits at a floor.

In [ ]:
# 1) Temperature — C_ece must move while AP stays flat.
!python tools/synthetic_controls.py --control temperature \
  --ann data/underwater/annotations/instances_val.json \
  --results runs/diag/underwater/owlv2/results_global.json \
  --out runs/controls/underwater_owlv2_temperature.csv

In [ ]:
# 2) Vocabulary — random vs semantic distractors.
#    With only ~6 native classes the distractor pool is small, so we cap the ladder.
#    semantic >= random at matched count => S tracks confusability, not just size.
for mode in ['random', 'semantic']:
    !python tools/synthetic_controls.py --control vocab --distractor {mode} \
      --ann data/underwater/annotations/instances_val.json --imgs "{IMG_DIR}" \
      --model owlv2 --weights google/owlv2-base-patch16-ensemble \
      --out runs/controls/underwater_vocab_{mode}.csv --limit 300 --device cuda:0

In [ ]:
# 3) Blur — L must rise. Unlike medical, L should start well below 1 here,
#    so this control has room to move and is actually informative.
!python tools/synthetic_controls.py --control blur \
  --ann data/underwater/annotations/instances_val.json --imgs "{IMG_DIR}" \
  --sam_weights mobile_sam.pt --out runs/controls/underwater_blur.csv --limit 200 --device cuda:0

In [ ]:
import pandas as pd, os
for name in ['underwater_owlv2_temperature', 'underwater_vocab_random',
             'underwater_vocab_semantic', 'underwater_blur']:
    p = f'runs/controls/{name}.csv'
    if os.path.exists(p):
        print(f'== {name} =='); print(pd.read_csv(p).to_string(index=False), '\n')

## Three-domain read
Paste all three `fingerprints.csv` into the paper table. The claim the benchmark can
support depends on where underwater lands:

- **Non-degenerate, vocabulary-bound** (high S, moderate L) — aerial's pattern replicates
  on independent imagery. Strongest outcome: the S axis generalizes.
- **Non-degenerate, localization-bound** (high L, low S) — two distinct failure modes each
  seen twice, on four datasets. The decomposition discriminates. Also strong.
- **Mixed / intermediate on both axes** — best case for the *protocol*: shows the
  fingerprint resolving a domain that a single mAP number would call "just bad".
- **Degenerate** — prediction falsified; report the uniform-failure finding honestly and
  keep the paper at two domains.

Do not adjust the framing after seeing the number and then present it as a prior
expectation — the prediction above is timestamped in git for exactly this reason.